In [ ]:
class OrderRequest:
    def __init__(self, symbol, side, quantity, price):
        self.symbol = symbol      # symbol을 받아서
        self.side = side          # self.symbol에 저장
        self.quantity = quantity  # 매번 이걸 반복...
        self.price = price

In [1]:
from dataclasses import dataclass

# 기존 방식
class OrderRequest1:
    def __init__(self, symbol, side, quantity, price):
        self.symbol = symbol
        self.side = side
        self.quantity = quantity
        self.price = price

# @dataclass 방식
@dataclass
class OrderRequest2:
    symbol: str
    side: str
    quantity: int
    price: int

# 둘 다 똑같이 사용해요
o1 = OrderRequest1("005930", "buy", 10, 70000)
o2 = OrderRequest2("005930", "buy", 10, 70000)

print(o1.symbol)   # 005930
print(o2.symbol)   # 005930

# 여기서 차이가 나요
print(o1)          # 어떻게 나오나요?
print(o2)          # 어떻게 나오나요?

005930
005930
OrderRequest2(symbol='005930', side='buy', quantity=10, price=70000)


`print(o1)` → `<__main__.OrderRequest1 object at 0x107b024e0>`
이건 "OrderRequest1의 인스턴스인데, 메모리 어딘가에 있어요" 라는 의미.

`print(o2)` → `OrderRequest2(symbol='005930', side='buy', quantity=10, price=70000)`
이건 내용물이 바로 보여서, 디버깅할 때 매우 편리.

이 차이가 생기는 이유는 `@dataclass`가 자동으로 만들어주는 것들 때문:

| 자동 생성 | 역할 |
|---|---|
|`__init__` | 생성자 자동 생성 (반복 코드 제거) |
|`__repr__` | print() 했을 때 내용물 보여줌 |
|`__eq__` | o1 == o2 비교 가능 |


In [ ]:
o3 = OrderRequest2("005930", "buy", 10, 70000)
o4 = OrderRequest2("005930", "buy", 10, 70000)

print(o3 == o4)  # @dataclass 없으면 False, 있으면?

# @dataclass 없이 일반 클래스로 만들면 o3 == o4는 False. 내용물이 똑같아도 메모리 주소가 다른 두 개의 객체이기 때문에.

True


In [ ]:
@dataclass
class OrderRequest:
    symbol: str
    side: str
    quantity: int
    price: int
    order_type: str = "00"        # 기본값 있음
    account_no: str = ""          # 기본값 있음

    def __post_init__(self) -> None:    # __init__ 실행 직후 자동으로 호출됨. 주로 데이터 검증에 씀.
        if self.quantity <= 0:
            raise ValueError("quantity must be > 0")

In [ ]:
from dataclasses import dataclass

@dataclass
class OrderRequest:
    symbol: str
    quantity: int
    price: int
    order_type: str = "00"

    def __post_init__(self):
        if self.quantity <= 0:
            raise ValueError("quantity must be > 0")

# 정상
o1 = OrderRequest("005930", 10, 70000)
print(o1)

# 비정상 — 어떤 일이 일어날까요?
o2 = OrderRequest("005930", -5, 70000)  # 실행하면 ValueError: quantity must be > 0 오류 발생. 객체가 만들어지는 순간 검증이 실행돼서 바로 막아버림. 잘못된 데이터가 시스템 안으로 아예 못 들어오게 하는 것.

OrderRequest(symbol='005930', quantity=10, price=70000, order_type='00')


ValueError: quantity must be > 0

In [ ]:
# 주문을 넣는 개발자가 코드를 보고 있어요
o1 = OrderRequest("005930", 10, 70000, "00")   # "00"이 뭐지?
o2 = OrderRequest("005930", 10, 70000, "01")   # "01"은?
o3 = OrderRequest("005930", 10, 70000, "99")   # 이건 유효한 값인가?

In [4]:
# "00", "01" 같은 코드는 외워야만 알 수 있는 것들이라 코드 읽기 어려움. 오타를 내도 오류가 안 나고 그냥 잘못된 주문이 들어가 버림.
# Enum은 이 문제를 해결.

from enum import Enum

class OrderType(str, Enum):
    LIMIT       = "00"   # 지정가
    MARKET      = "01"   # 시장가
    PRE_MARKET  = "05"   # 장전시간외
    POST_MARKET = "06"   # 장후시간외

In [5]:
o1 = OrderRequest("005930", 10, 70000, OrderType.LIMIT)    # "00"이 뭔지 몰라도 됨
o2 = OrderRequest("005930", 10, 70000, OrderType.MARKET)   # 오타 내면 즉시 오류
o3 = OrderRequest("005930", 10, 70000, OrderType.WRONG)    # 이건 어떻게 될까요?

AttributeError: type object 'OrderType' has no attribute 'WRONG'

In [6]:
from enum import Enum

class OrderType(str, Enum):
    LIMIT       = "00"
    MARKET      = "01"
    PRE_MARKET  = "05"
    POST_MARKET = "06"

# 이것들을 출력해봐요
print(OrderType.LIMIT)          # 어떻게 나올까요?
print(OrderType.LIMIT.value)    # 이건요?
print(OrderType.MARKET == "01") # 이건요? (str을 상속받았기 때문에...)

# 잘못된 값 접근
print(OrderType.WRONG)

OrderType.LIMIT
00
True


AttributeError: type object 'OrderType' has no attribute 'WRONG'

# Enum을 쓰는 이유 세 가지
| 문제 | Enum으로 해결 |
|---|---|
|`"00"`이 뭔지 외워야 함 | `OrderType.LIMIT` 으로 의미가 명확해짐 |
| 오타를 내도 오류 없이 통과 | 정의된 값 외에는 `AttributeError` |
| API에 문자열로 넘겨야 함 | `str` 상속으로 `.value` 없이 바로 사용 |
